In [30]:
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile as tiff

from skimage.filters import gaussian, threshold_otsu
from skimage.morphology import (
    remove_small_objects,
    remove_small_holes,
    disk,
    opening,
)
from skimage.measure import label, regionprops_table
from skimage.exposure import rescale_intensity
from skimage.restoration import rolling_ball

In [31]:
# paths

BASE_DIR = Path.cwd()

CONDENSATE_RAW_PATH = BASE_DIR / "data" / "raw_mask_condensates" / "C2-raw_stack_sample2_5.ome.tif"
CONDENSATE_MASK_PATH = BASE_DIR / "data" / "raw_mask_condensates" / "C2-raw_stack_sample2_5.ome_cp_masks.tif"

NUCLEI_RAW_PATH = BASE_DIR / "data" / "raw_mask_nuclei" / "C1-raw_stack_sample2_5.tif"
NUCLEI_MASK_PATH = BASE_DIR / "data" / "raw_mask_nuclei" / "C1-raw_stack_sample2_5.ome_cp_masks.tif"

OUTPUT_DIR = BASE_DIR / "outputs" / "python_native_segmentation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Versioned directories
COND_BASE = OUTPUT_DIR / "condensates"
V1_DIR = COND_BASE / "v1"
V2_DIR = COND_BASE / "v2"

V1_DIR.mkdir(parents=True, exist_ok=True)
V2_DIR.mkdir(parents=True, exist_ok=True)

In [32]:
print("Checking file existence...")
print("Condensate raw exists:", CONDENSATE_RAW_PATH.exists())
print("Condensate Cellpose mask exists:", CONDENSATE_MASK_PATH.exists())
print("Nuclei raw exists:", NUCLEI_RAW_PATH.exists())
print("Nuclei Cellpose mask exists:", NUCLEI_MASK_PATH.exists())

condensate_raw = tiff.imread(CONDENSATE_RAW_PATH)
condensate_mask = tiff.imread(CONDENSATE_MASK_PATH)

nuclei_raw = tiff.imread(NUCLEI_RAW_PATH)
nuclei_mask = tiff.imread(NUCLEI_MASK_PATH)

print("\nLoaded image shapes:")
print("Raw Condensate Shape:", condensate_raw.shape)
print("Mask Condensate Shape:", condensate_mask.shape)
print("Raw Nuclei Shape:", nuclei_raw.shape)
print("Mask Nuclei Shape:", nuclei_mask.shape)

Checking file existence...
Condensate raw exists: True
Condensate Cellpose mask exists: True
Nuclei raw exists: True
Nuclei Cellpose mask exists: True

Loaded image shapes:
Raw Condensate Shape: (55, 1800, 1800)
Mask Condensate Shape: (55, 1800, 1800)
Raw Nuclei Shape: (55, 1800, 1800)
Mask Nuclei Shape: (55, 1800, 1800)


In [33]:
# Condensates
COND_GAUSSIAN_SIGMA = 1.5
COND_ROLLING_BALL_RADIUS = 40
COND_MIN_OBJECT_SIZE = 30
COND_MIN_HOLE_SIZE = 8


# Nuclei
NUC_GAUSSIAN_SIGMA = 2.0
NUC_ROLLING_BALL_RADIUS = 50
NUC_MIN_OBJECT_SIZE = 200
NUC_MIN_HOLE_SIZE = 200

In [34]:
# =========================================================
# SEGMENTATION FUNCTIONS
# =========================================================
def segment_condensates(img):
    img = img.astype(np.float32)

    background = rolling_ball(img, radius=COND_ROLLING_BALL_RADIUS)
    corrected = img - background
    corrected[corrected < 0] = 0

    smoothed = gaussian(corrected, sigma=COND_GAUSSIAN_SIGMA)
    smoothed = rescale_intensity(smoothed, in_range="image", out_range=(0, 1))

    thresh = threshold_otsu(smoothed)
    binary = smoothed > thresh

    binary = remove_small_objects(binary, min_size=COND_MIN_OBJECT_SIZE)
    binary = remove_small_holes(binary, area_threshold=COND_MIN_HOLE_SIZE)
    binary = opening(binary, footprint=disk(1))
    binary = remove_small_objects(binary, min_size=COND_MIN_OBJECT_SIZE)

    labels = label(binary)

    return labels, binary


def segment_nuclei(img):
    img = img.astype(np.float32)

    background = rolling_ball(img, radius=NUC_ROLLING_BALL_RADIUS)
    corrected = img - background
    corrected[corrected < 0] = 0

    smoothed = gaussian(corrected, sigma=NUC_GAUSSIAN_SIGMA)
    smoothed = rescale_intensity(smoothed, in_range="image", out_range=(0, 1))

    thresh = threshold_otsu(smoothed)
    binary = smoothed > thresh

    binary = remove_small_objects(binary, min_size=NUC_MIN_OBJECT_SIZE)
    binary = remove_small_holes(binary, area_threshold=NUC_MIN_HOLE_SIZE)
    binary = opening(binary, footprint=disk(2))
    binary = remove_small_objects(binary, min_size=NUC_MIN_OBJECT_SIZE)

    labels = label(binary)

    return labels, binary


In [35]:
# =========================================================
# RUN CONDENSATE SEGMENTATION
# =========================================================
print("\nRunning condensate segmentation...")

cond_binary_stack = []
cond_label_stack = []
cond_measurements = []

for z in range(condensate_raw.shape[0]):
    img = condensate_raw[z]

    labels, binary = segment_condensates(img)

    cond_binary_stack.append(binary.astype(np.uint8))
    cond_label_stack.append(labels.astype(np.int32))

    props = regionprops_table(
        labels,
        intensity_image=img,
        properties=["label", "area", "centroid", "mean_intensity"],
    )

    df = pd.DataFrame(props)
    df["z"] = z
    cond_measurements.append(df)

    print(f"Condensate slice {z}: {labels.max()} objects")

cond_binary_stack = np.stack(cond_binary_stack)
cond_label_stack = np.stack(cond_label_stack)
condensate_df = pd.concat(cond_measurements, ignore_index=True)


Running condensate segmentation...
Condensate slice 0: 1447 objects
Condensate slice 1: 1383 objects
Condensate slice 2: 1 objects
Condensate slice 3: 1423 objects
Condensate slice 4: 1448 objects
Condensate slice 5: 1413 objects
Condensate slice 6: 1458 objects
Condensate slice 7: 1452 objects
Condensate slice 8: 1482 objects
Condensate slice 9: 1518 objects
Condensate slice 10: 1490 objects
Condensate slice 11: 1411 objects
Condensate slice 12: 28 objects
Condensate slice 13: 32 objects
Condensate slice 14: 34 objects
Condensate slice 15: 26 objects
Condensate slice 16: 27 objects
Condensate slice 17: 25 objects
Condensate slice 18: 22 objects
Condensate slice 19: 25 objects
Condensate slice 20: 25 objects
Condensate slice 21: 18 objects
Condensate slice 22: 19 objects
Condensate slice 23: 18 objects
Condensate slice 24: 16 objects
Condensate slice 25: 14 objects
Condensate slice 26: 11 objects
Condensate slice 27: 9 objects
Condensate slice 28: 7 objects
Condensate slice 29: 5 obje

In [38]:
# =========================================================
# RUN NUCLEI SEGMENTATION
# =========================================================
print("\nRunning nuclei segmentation...")

nuc_binary_stack = []
nuc_label_stack = []
nuc_measurements = []

for z in range(nuclei_raw.shape[0]):
    img = nuclei_raw[z]

    labels, binary = segment_nuclei(img)

    nuc_binary_stack.append(binary.astype(np.uint8))
    nuc_label_stack.append(labels.astype(np.int32))

    props = regionprops_table(
        labels,
        intensity_image=img,
        properties=["label", "area", "centroid", "mean_intensity"],
    )

    df = pd.DataFrame(props)
    df["z"] = z
    nuc_measurements.append(df)

    print(f"Nuclei slice {z}: {labels.max()} objects")

nuc_binary_stack = np.stack(nuc_binary_stack)
nuc_label_stack = np.stack(nuc_label_stack)
nuclei_df = pd.concat(nuc_measurements, ignore_index=True)


Running nuclei segmentation...
Nuclei slice 0: 25 objects
Nuclei slice 1: 19 objects
Nuclei slice 2: 28 objects
Nuclei slice 3: 14 objects
Nuclei slice 4: 19 objects
Nuclei slice 5: 35 objects
Nuclei slice 6: 65 objects
Nuclei slice 7: 127 objects
Nuclei slice 8: 149 objects
Nuclei slice 9: 163 objects
Nuclei slice 10: 173 objects
Nuclei slice 11: 174 objects
Nuclei slice 12: 178 objects
Nuclei slice 13: 173 objects
Nuclei slice 14: 170 objects
Nuclei slice 15: 154 objects
Nuclei slice 16: 134 objects
Nuclei slice 17: 124 objects
Nuclei slice 18: 122 objects
Nuclei slice 19: 111 objects
Nuclei slice 20: 101 objects
Nuclei slice 21: 108 objects
Nuclei slice 22: 120 objects
Nuclei slice 23: 125 objects
Nuclei slice 24: 125 objects
Nuclei slice 25: 145 objects
Nuclei slice 26: 152 objects
Nuclei slice 27: 156 objects
Nuclei slice 28: 167 objects
Nuclei slice 29: 166 objects
Nuclei slice 30: 161 objects
Nuclei slice 31: 147 objects
Nuclei slice 32: 137 objects
Nuclei slice 33: 122 objects

In [ ]:
# =========================================================
# SAVE FUNCTION
# =========================================================
def save_version(version_dir):
    print(f"\nSaving to {version_dir}...")

    tiff.imwrite(version_dir / "binary_masks.tif", cond_binary_stack)
    tiff.imwrite(version_dir / "label_masks.tif", cond_label_stack)
    condensate_df.to_csv(version_dir / "measurements.csv", index=False)


# =========================================================
# CHOOSE VERSION TO SAVE
# =========================================================
VERSION = "v2"  # change to "v2" after tuning

if VERSION == "v1":
    save_version(V1_DIR)
elif VERSION == "v2":
    save_version(V2_DIR)


# =========================================================
# SAVE NUCLEI (no versioning needed)
# =========================================================
tiff.imwrite(OUTPUT_DIR / "nuclei_binary_masks.tif", nuc_binary_stack)
tiff.imwrite(OUTPUT_DIR / "nuclei_label_masks.tif", nuc_label_stack)

nuclei_df.to_csv(
    OUTPUT_DIR / "python_native_nuclei_measurements.csv",
    index=False
)

print("\nDone.")


Saving to c:\storage\code\research\outputs\python_native_segmentation\condensates\v2...

Done.
